# AQ26 33A — Google Drive Evidence Lake Exporter

This notebook mounts Google Drive, copies LAQN provider outputs into the controlled AQ26 evidence lake, writes checksums/manifests, and prepares a compact website index.

Run this after the LAQN v3.5 GitHub workflow has produced `outputs/31_laqn`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Adjust these if your repo is in a different folder.
from pathlib import Path

DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/SCC_NEXUS_100GB_ROOT/PROJECTS/AirQuality26')
REPO_ROOT = DRIVE_PROJECT_ROOT / 'repo' / 'AirQuality26_v2'

print('DRIVE_PROJECT_ROOT exists:', DRIVE_PROJECT_ROOT.exists(), DRIVE_PROJECT_ROOT)
print('REPO_ROOT exists:', REPO_ROOT.exists(), REPO_ROOT)
print('LAQN outputs exist:', (REPO_ROOT / 'outputs/31_laqn').exists(), REPO_ROOT / 'outputs/31_laqn')


## Option A: run the exporter from inside your repo

Use this when you have copied the patch files into the repository.

In [ ]:
!python "{REPO_ROOT}/scripts/aq26_drive_evidence_exporter.py" \
  --repo-root "{REPO_ROOT}" \
  --drive-root "{DRIVE_PROJECT_ROOT}" \
  --provider laqn \
  --source-subdir outputs/31_laqn \
  --site-public "{REPO_ROOT}/site_public" \
  --copy-mode copy \
  --commit-site-ready


## Optional: controlled tiny LAQN historical harvest

Only run this after confirming `site_species_london_rows.csv` exists. Keep the first run tiny.

In [ ]:
SITE_SPECIES = REPO_ROOT / 'outputs/31_laqn/site_species_london_rows.csv'
OUT_ROOT = DRIVE_PROJECT_ROOT / 'data/raw/laqn'
print('site species csv:', SITE_SPECIES.exists(), SITE_SPECIES)
print('out root:', OUT_ROOT)


In [ ]:
# Tiny test only: max 3 site/species pairs for one day.
!python "{REPO_ROOT}/scripts/aq26_laqn_historical_harvest_checkpointed.py" \
  --site-species-csv "{SITE_SPECIES}" \
  --out-root "{OUT_ROOT}" \
  --start-date 2024-07-22 \
  --end-date 2024-07-23 \
  --max-pairs 3 \
  --sleep-seconds 1


## Verify outputs

In [ ]:
import json, os
latest = DRIVE_PROJECT_ROOT / 'site_ready/providers/laqn/latest_evidence_lake_index.json'
print('latest evidence lake index:', latest.exists(), latest)
if latest.exists():
    print(latest.read_text()[:2000])

repo_site_index = REPO_ROOT / 'site_public/data/providers/laqn/evidence_lake/latest_index.json'
print('repo website index:', repo_site_index.exists(), repo_site_index)
if repo_site_index.exists():
    print(repo_site_index.read_text()[:2000])


## Commit website index from Colab, if you are using Drive-mounted git

Only run this if your repo folder is a real git checkout with write access.

In [ ]:
# Optional. Uncomment after checking the files.
# %cd {REPO_ROOT}
# !git status --short
# !git add site_public/data/providers/laqn/evidence_lake/latest_index.json site_public/assets/aq26_evidence_lake.js
# !git commit -m 'Add AQ26 evidence lake website index'
# !git push
